# 실습 1 · 중심 벡터 추천 — 쇼핑몰 구매 이력

장표 18(평균) · 20(중심 벡터 검색/추천). **구매한 상품 벡터의 평균 = 취향 벡터** 로 다음 구매를 맞히는지 실제 쇼핑몰 데이터로 잰다.

- 데이터 : Amazon Reviews 2023 · Video_Games 5-core. 사용자마다 구매 이력(시간순) + **가려 둔 마지막 구매**
- 이 노트북만으로 끝난다 : 받기 → 테이블 → 적재 → 임베딩 → 실습. 표본 사용자 `N_USERS` 명이 산 상품만 임베딩한다(500명 = 상품 3,261개 · 강사 PC 단독 약 20분)
- 채점 : 가려 둔 상품이 추천 상위 10 안이면 Hit@10, MRR = 그 순위의 역수 평균(10위 밖은 0). 이미 산 상품은 추천에서 뺀다. 후보 = 테이블의 상품 전체

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 준비 — DB · 임베딩 서버 · 도구 (01~04 공통) |
| 3 | 테이블 만들기 — `lab3.shop_items` · `lab3.shop_purchases` |
| 4 | 받기 · 표 — 상품 메타 · 사용자 이력 → 표본 사용자 |
| 5 | 적재 |
| 6 | 임베딩 · HNSW 인덱스 |
| 7 | 불러오기 · 채점 도구 |
| 8 | 추천 SQL — 한 사용자의 취향 벡터 `avg(embedding)` → 가까운 상품 |
| 9 | 추천 API — HNSW 후보 → 원본 재정렬 (4-1 실습5 방식) · 속도와 적중 |
| 10 | 추천 방식 비교 — 인기순 · 마지막 구매 · 평균 · 최근 가중 · 분류별 평균, 각각 공통 성분 제거 전후 |
| 11 | 시간 순서를 뺀 채점 — 이력 중 무작위 하나를 가림 : 평균 vs 한 개 |
| 12 | ‖M‖(응집도)과 추천 적중 |
| 13 | 중심 벡터 : 코사인이든 내적이든 순위는 같다 |
| 14 | 정리 |

### 셀 1 · 설정
- `N_USERS` : 표본 사용자 수 → 이들의 이력 · 마지막 구매에 나온 상품만 적재 · 임베딩. `None` = 전부(사용자 9만여 명 · 상품 2만 5천여 개 · 임베딩 수 시간)
- `N_API` · `CANDIDATES` : 셀 9 에서 SQL 로 추천해 볼 사용자 수 · HNSW 후보 수

In [ ]:
# 셀 1 · 설정
DB_URL       = "postgresql://lab:lab@db:5432/lab"       # 4-1 도커의 DB
LMSTUDIO_URL = "http://host.docker.internal:1234/v1"    # LM Studio 서버 주소 (4-1 실습1 셀 1 과 같게)
EMBED_MODEL  = "text-embedding-qwen3-embedding-8b"      # LM Studio 에 로드된 임베딩 모델 이름
CACHE        = "/cache/hf/lab3"                         # 받은 원본 저장 위치 = 도커 볼륨 hfcache
BATCH_SIZE   = 16                                       # 한 번에 LM Studio 로 보내는 글 수
MAX_CHARS    = 1500                                     # 이보다 긴 글은 잘라서 넣는다
SEED         = 42
RESET        = False    # True : 이 실습의 테이블을 지우고 다시 만든다(표본 크기를 바꿨을 때). 이미 계산한 벡터는 캐시에서 돌아온다

N_USERS     = 500        # 표본 사용자 수 (None = 전부)
MIN_HIST    = 2          # 이력이 이보다 짧은 사용자는 뺀다
N_EVAL      = 5000       # 셀 10~ 에서 채점할 사용자 수 상한 (N_USERS 가 크면 이만큼 무작위로)
RECENT_HALF = 2          # 최근 가중 : 한 칸 과거로 갈 때마다 가중치 × 0.5^(1/2)
N_API       = 100        # 셀 9 : SQL 로 추천해 볼 사용자 수
CANDIDATES  = [50, 200, 800]   # 셀 9 : HNSW 후보 수

### 셀 2 · 준비 — DB · 임베딩 서버 · 도구
01~04 실습의 셀 2 는 모두 같다(각 실습을 따로 돌릴 수 있게 노트북마다 들어 있다).
- DB 연결(4-1 과 같은 pgvector DB, 스키마 `lab3`) · LM Studio 확인
- **임베딩 캐시** `lab3.emb_cache` : 키 = (모델 이름, 글의 md5) → 벡터. 같은 글은 한 번만 계산한다(예 : 실습 3 에서 계산한 판례는 실습 4 에서 다시 계산하지 않음). 실습 테이블을 지워도 남고, 지우려면 `TRUNCATE lab3.emb_cache`
- `embed_table(테이블)` : 4-1 실습1 셀 7 과 같은 배치 임베딩. ① 캐시에서 먼저 채우고 ② 남은 행을 `BATCH_SIZE` 개씩 LM Studio 로 → `UPDATE` + 캐시 저장. 멈췄다 다시 실행하면 빈 행부터
- `load(SQL)` : 마지막 열이 벡터인 SELECT → (표, 벡터 행렬 · 길이 1)
- `ci(값들)` : 단위별 차이의 평균과 부트스트랩 95% 신뢰구간 — 표본이 작아 차이가 우연인지 함께 본다(구간이 0 을 포함하면 "차이가 있다고 말할 수 없음")

In [ ]:
# 셀 2 · 준비
import json, time, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg
import requests
from pgvector.psycopg import register_vector

plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False, "figure.dpi": 110})
pd.set_option("display.max_colwidth", 70)
C = Path(CACHE); (C / "raw").mkdir(parents=True, exist_ok=True)
HF = "https://huggingface.co/datasets"
rng = np.random.default_rng(SEED)

conn = psycopg.connect(DB_URL, autocommit=True)          # 문장마다 바로 반영 (여러 문장을 묶을 때만 conn.transaction())
register_vector(conn)
conn.execute("CREATE SCHEMA IF NOT EXISTS lab3")
conn.execute("""
    CREATE TABLE IF NOT EXISTS lab3.emb_cache (          -- 임베딩 캐시 (01~04 공용)
        model     text NOT NULL,                         -- 임베딩 모델 이름 (모델이 다르면 벡터도 다르다)
        text_md5  text NOT NULL,                         -- md5(글)
        embedding vector(4096) NOT NULL,
        PRIMARY KEY (model, text_md5))""")


def one(q, params=None):
    """SQL 결과의 첫 행 첫 값"""
    row = conn.execute(q, params).fetchone()
    return row[0] if row else None


def sql(q, params=None):
    """SQL 결과 → 표"""
    with conn.cursor() as cur:
        cur.execute(q, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])


def unit(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


def ci(x, n=1000):
    """단위(사용자 · 질의 · 글 …)별 값 x 의 평균과 부트스트랩 95% 신뢰구간. 구간이 0 을 포함하지 않으면 ✔ (차이가 우연이라고 보기 어려움)"""
    x = np.asarray(x, float); r = np.random.default_rng(SEED)
    b = [x[r.integers(0, len(x), len(x))].mean() for _ in range(n)]
    lo, hi = np.quantile(b, [.025, .975])
    return f"{x.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]" + (" ✔" if lo > 0 or hi < 0 else "")


def load(q, params=None):
    """마지막 열이 벡터인 SELECT → (표, 벡터 행렬 float32 · 길이 1). 이진 전송이라 벡터가 바로 numpy 로 온다"""
    with conn.cursor(binary=True) as cur:
        cur.execute(q, params)
        rows = cur.fetchall()
        names = [c.name for c in cur.description]
    assert rows and all(r[-1] is not None for r in rows), "임베딩이 빈 행이 있다 → 셀 6(임베딩)을 끝까지"
    return pd.DataFrame([r[:-1] for r in rows], columns=names[:-1]), unit(np.stack([r[-1] for r in rows]).astype(np.float32))


def download(url, name):
    """볼륨에 없을 때만 받는다(.part 로 받고 끝나면 이름 변경)"""
    p = C / "raw" / name
    if not p.exists():
        t = time.time(); tmp = p.with_suffix(p.suffix + ".part")
        urllib.request.urlretrieve(url, tmp); tmp.rename(p)
        print(f"  받음 {name} {p.stat().st_size / 1e6:.1f}MB · {time.time() - t:.0f}s")
    return p


def parquet_urls(ds, config, split):
    r = requests.get(f"https://huggingface.co/api/datasets/{ds}/parquet/{config}/{split}", timeout=60)
    r.raise_for_status(); return r.json()


def all_parquet_urls(ds, config="default"):
    """그 구성의 모든 분할(train · test …) parquet 주소"""
    r = requests.get(f"https://huggingface.co/api/datasets/{ds}/parquet", timeout=60); r.raise_for_status()
    return [u for us in r.json()[config].values() for u in us]


def clean(d):
    """글 다듬기 : NUL 제거 · 앞뒤 공백 · MAX_CHARS 에서 자르기"""
    d = d.reset_index(drop=True)
    d["text"] = d["text"].astype(str).str.replace("\x00", "", regex=False).str.strip().str.slice(0, MAX_CHARS)
    d["label"] = d["label"].astype(str)
    return d


def per_label(d, k):
    """라벨마다 k 개씩 무작위. 같은 SEED 면 작은 k 의 표본은 큰 k 표본의 앞부분이다(→ 캐시가 그대로 맞는다)"""
    return pd.concat([g.sample(min(len(g), k), random_state=SEED) for _, g in d.groupby("label")]).reset_index(drop=True)


def embed(texts):
    """LM Studio /v1/embeddings. 실패하면 5 · 10 · 20초 뒤 다시"""
    for wait in [5, 10, 20, None]:
        try:
            r = requests.post(f"{LMSTUDIO_URL}/embeddings", json={"model": EMBED_MODEL, "input": texts}, timeout=600)
            r.raise_for_status()
            return [np.asarray(d["embedding"], dtype=np.float32) for d in sorted(r.json()["data"], key=lambda d: d["index"])]
        except Exception as e:
            if wait is None: raise
            print(f"  ! 임베딩 실패({type(e).__name__}) → {wait}s 뒤 다시"); time.sleep(wait)


def embed_table(table):
    """빈 embedding 채우기 : ① 캐시(같은 모델 · 같은 글) → ② 남은 행을 BATCH_SIZE 개씩 LM Studio 로 · UPDATE + 캐시 저장"""
    t0 = time.time()
    hit = conn.execute(f"""UPDATE {table} t SET embedding = c.embedding FROM lab3.emb_cache c
                           WHERE t.embedding IS NULL AND c.model = %s AND c.text_md5 = md5(t.text)""", (EMBED_MODEL,)).rowcount
    todo = one(f"SELECT count(*) FROM {table} WHERE embedding IS NULL")
    print(f"[{table}] 캐시에서 {hit:,}건 · 새로 임베딩 {todo:,}건", flush=True)
    done, nb, t1 = 0, 0, time.time()
    while True:
        rows = conn.execute(f"SELECT id, text FROM {table} WHERE embedding IS NULL ORDER BY id LIMIT %s", (BATCH_SIZE,)).fetchall()
        if not rows:
            break
        vecs = embed([t for _, t in rows])
        with conn.transaction(), conn.cursor() as cur:
            cur.executemany(f"UPDATE {table} SET embedding = %s WHERE id = %s", [(v, i) for (i, _), v in zip(rows, vecs)])
            cur.executemany("INSERT INTO lab3.emb_cache VALUES (%s, md5(%s), %s) ON CONFLICT DO NOTHING",
                            [(EMBED_MODEL, t, v) for (_, t), v in zip(rows, vecs)])
        done += len(rows); nb += 1
        if nb % 25 == 0 or done >= todo:
            el = time.time() - t1; rate = done / el
            print(f"  {done:6,}/{todo:,}  {rate:5.1f}건/s  경과 {el / 60:5.1f}분  남은 시간 약 {max(todo - done, 0) / rate / 60:5.1f}분", flush=True)
    print(f"[{table}] 끝 · 캐시 {hit:,}건 + 새로 {done:,}건 · {time.time() - t0:.0f}s")


try:
    models = [m["id"] for m in requests.get(f"{LMSTUDIO_URL}/models", timeout=10).json()["data"]]
except requests.exceptions.RequestException:
    raise ConnectionError(f"LM Studio 에 연결할 수 없음 : {LMSTUDIO_URL}\n"
                          "  ① LM Studio 의 Developer 탭에서 서버가 켜져 있는지 확인\n"
                          "  ② 포트가 다르면 셀 1 의 LMSTUDIO_URL 을 고치고 셀 1 부터 다시") from None
assert EMBED_MODEL in models, f"셀 1 의 EMBED_MODEL 을 이 목록의 이름으로 바꾼다 : {models}"
print("pgvector", one("SELECT extversion FROM pg_extension WHERE extname = 'vector'"), "· 임베딩", embed(["확인"])[0].size, "차원",
      f"· 캐시 {one('SELECT count(*) FROM lab3.emb_cache WHERE model = %s', (EMBED_MODEL,)):,}건")

### 셀 3 · 테이블 만들기
4-1 의 `docs` 와 같은 규칙 : 원본 `embedding vector(4096)` + 앞 128차원을 다시 길이 1로 만든 `emb128`(생성 컬럼, HNSW 는 여기에).
구매 이력은 사용자마다 시간순 `seq` 로, 마지막 구매는 `is_target = true` 로 가려 둔다.

In [ ]:
# 셀 3 · 테이블
if RESET:
    conn.execute("DROP TABLE IF EXISTS lab3.shop_purchases, lab3.shop_items")
conn.execute("""
    CREATE TABLE IF NOT EXISTS lab3.shop_items (
        id        bigserial PRIMARY KEY,
        asin      text UNIQUE NOT NULL,          -- 아마존 상품 코드 (parent_asin)
        title     text,
        category  text,                          -- 분류 2단계 (대부분 플랫폼 : PC · Nintendo Switch …)
        text      text NOT NULL,                 -- 임베딩할 글 = 제목 | 분류 | 플랫폼 | 특징 3개
        embedding vector(4096),                  -- Qwen3-Embedding-8B · 지시문 없이
        emb128    vector(128) GENERATED ALWAYS AS (
                      l2_normalize(subvector(embedding, 1, 128))::vector(128)) STORED)""")
conn.execute("""
    CREATE TABLE IF NOT EXISTS lab3.shop_purchases (
        user_id   text    NOT NULL,
        seq       int     NOT NULL,              -- 시간순 0, 1, 2 …
        item_id   bigint  NOT NULL REFERENCES lab3.shop_items(id) ON DELETE CASCADE,
        is_target boolean NOT NULL,              -- 가려 둔 마지막 구매 (맞힐 대상)
        PRIMARY KEY (user_id, seq))""")
sql("SELECT (SELECT count(*) FROM lab3.shop_items) AS 상품, (SELECT count(*) FROM lab3.shop_purchases) AS 구매")

### 셀 4 · 받기 · 표
상품 메타(약 90MB)와 5-core 사용자 이력을 받아(볼륨에 없을 때만) 표로 만든다.
이력이 `MIN_HIST` 개 이상이고 마지막 구매의 메타가 있는 사용자 중 `N_USERS` 명을 뽑고, 그들이 산 상품만 남긴다.

In [ ]:
# 셀 4 · 받기 · 표
def shop():
    """Amazon Reviews 2023 · Video_Games 5-core : (상품 표, 사용자 표)
    상품 글 = 제목 | 분류 | 플랫폼 | 특징 3개 · category = 분류 2단계(대부분 플랫폼) · 사용자 = 이력(시간순, 공백 구분) + 가려 둔 마지막 구매"""
    base = f"{HF}/McAuley-Lab/Amazon-Reviews-2023/resolve/main"
    users = pd.read_csv(download(f"{base}/benchmark/5core/last_out_w_his/Video_Games.test.csv", "shop_test.csv"))
    need = set(users["parent_asin"]) | set(" ".join(users["history"].fillna("")).split())
    rows = []
    with open(download(f"{base}/raw/meta_categories/meta_Video_Games.jsonl", "shop_meta.jsonl"), encoding="utf-8") as f:
        for line in f:
            m = json.loads(line)
            if m["parent_asin"] not in need: continue
            det, cats = m.get("details") or {}, m.get("categories") or []
            feats = " ; ".join((m.get("features") or [])[:3])
            text = " | ".join(x for x in [m.get("title") or "", "category: " + " > ".join(cats),
                                          f"platform: {det.get('Platform')}" if det.get("Platform") else "", feats] if x)
            rows.append({"asin": m["parent_asin"], "text": text, "label": cats[1] if len(cats) > 1 else "?", "title": m.get("title") or ""})
    items = clean(pd.DataFrame(rows)).rename(columns={"label": "category"})
    return items, users[["user_id", "parent_asin", "history"]]


t = time.time()
items_all, users_all = shop()
have = set(items_all["asin"])
users_all = users_all.assign(hist=users_all["history"].fillna("").str.split().map(lambda h: [a for a in h if a in have]))
ok = users_all[users_all["parent_asin"].isin(have) & (users_all["hist"].str.len() >= MIN_HIST)]
users = ok if N_USERS is None else ok.sample(min(N_USERS, len(ok)), random_state=SEED)
need = set(users["parent_asin"]) | {a for h in users["hist"] for a in h}
items = items_all[items_all["asin"].isin(need)].reset_index(drop=True)
print(f"상품 메타 {len(items_all):,} · 사용자 {len(users_all):,} 중 이력 {MIN_HIST}개 이상 {len(ok):,}명 → 표본 {len(users):,}명 · 상품 {len(items):,}개 · {time.time() - t:.0f}s")
items[["asin", "category", "text"]].head(3)

### 셀 5 · 적재
4-1 실습1 과 같이 열마다 배열 하나씩 넘겨 `INSERT … SELECT FROM unnest(…)` 한 번으로. 이미 있으면 건너뛴다(다시 하려면 셀 1 `RESET = True`).
구매 이력은 상품 코드로 `shop_items` 와 조인해 `item_id` 를 채운다.

In [ ]:
# 셀 5 · 적재
if one("SELECT count(*) FROM lab3.shop_items"):
    print("이미 적재됨 → 건너뜀")
else:
    t = time.time()
    rows = [(u, s, a, False) for u, h in zip(users["user_id"], users["hist"]) for s, a in enumerate(h)] + \
           [(u, len(h), a, True) for u, h, a in zip(users["user_id"], users["hist"], users["parent_asin"])]
    with conn.transaction():
        conn.execute("""INSERT INTO lab3.shop_items (asin, title, category, text)
                        SELECT * FROM unnest(%s::text[], %s::text[], %s::text[], %s::text[])""",
                     [items[c].tolist() for c in ["asin", "title", "category", "text"]])
        conn.execute("""INSERT INTO lab3.shop_purchases (user_id, seq, item_id, is_target)
                        SELECT u.user_id, u.seq, i.id, u.is_target
                        FROM unnest(%s::text[], %s::int[], %s::text[], %s::bool[]) AS u(user_id, seq, asin, is_target)
                        JOIN lab3.shop_items i USING (asin)""", [list(c) for c in zip(*rows)])
    conn.execute("ANALYZE lab3.shop_items"); conn.execute("ANALYZE lab3.shop_purchases")
    print(f"적재 {time.time() - t:.1f}s")
sql("""SELECT count(*) AS 상품, (SELECT count(DISTINCT user_id) FROM lab3.shop_purchases) AS 사용자,
              (SELECT count(*) FROM lab3.shop_purchases WHERE NOT is_target) AS 이력_구매 FROM lab3.shop_items""")

### 셀 6 · 임베딩 · HNSW 인덱스
상품 글을 지시문 없이 임베딩한다. **Kernel → Interrupt** 로 멈춰도 채운 곳까지는 남는다(다시 실행하면 빈 행부터).
다 채운 뒤 `emb128` 에 HNSW 인덱스(셀 9 의 추천 API 용).

In [ ]:
# 셀 6 · 임베딩 · 인덱스
embed_table("lab3.shop_items")
conn.execute("CREATE INDEX IF NOT EXISTS shop_items_emb128_hnsw ON lab3.shop_items USING hnsw (emb128 vector_cosine_ops)")
conn.execute("ANALYZE lab3.shop_items")
sql("SELECT count(*) AS 상품, count(embedding) AS 임베딩, round(avg(vector_norm(embedding))::numeric, 4) AS 길이 FROM lab3.shop_items")

### 셀 7 · 불러오기 · 채점 도구
- 상품 벡터를 DB 에서 한 번 읽고(행 = id 순), 사용자별 이력 · 가려 둔 구매는 `array_agg(… ORDER BY seq)` 로 한 줄씩
- `score_users(프로필 함수)` : 사용자마다 프로필 벡터(들)를 만들어 전체 상품과 코사인 → 이력 상품 제외 → 가려 둔 상품의 순위
- 프로필이 여러 개(분류별 평균)면 상품마다 가장 가까운 프로필의 점수를 쓴다
- `Ec` = **공통 성분을 뺀 상품 벡터** : 상품 전체 평균 μ 를 빼고 다시 길이 1 로(장표 23). 실습 2 셀 12 에서 보듯 상품 벡터끼리는 코사인 평균이 0.5 쯤으로 한쪽에 쏠려 있어, 여러 상품을 평균하면 이 공통 방향이 커진다

In [ ]:
# 셀 7 · 불러오기 · 채점 도구
items, E = load("SELECT id, title, category, embedding FROM lab3.shop_items ORDER BY id")
pos = {i: k for k, i in enumerate(items["id"])}                    # shop_items.id → 행렬 행 번호
users = sql("""
    SELECT user_id, array_agg(item_id ORDER BY seq) FILTER (WHERE NOT is_target) AS hist,
           max(item_id) FILTER (WHERE is_target) AS target
    FROM lab3.shop_purchases GROUP BY user_id ORDER BY user_id""")   # 순서를 고정해야 표본 · 예시가 실행마다 같다
users = users[users["target"].notna() & (users["hist"].str.len() >= MIN_HIST)].copy()
users["hist"] = users["hist"].map(lambda h: [pos[i] for i in h])
users["target"] = users["target"].map(pos).astype(int)
U = users.sample(min(N_EVAL, len(users)), random_state=SEED).reset_index(drop=True)
pc = sql("SELECT item_id, count(*) AS n FROM lab3.shop_purchases WHERE NOT is_target GROUP BY item_id")   # 인기 = 이력에 나온 횟수
pop = np.zeros(len(items), np.float32); pop[pc["item_id"].map(pos).values] = pc["n"].values
lab = items["category"].values
mu = E.mean(axis=0)                                                # 공통 성분 = 상품 전체 평균
Ec = unit(E - mu)                                                  # 공통 성분을 뺀 상품 벡터 (길이 1)
print(f"상품 {len(items):,} · 사용자 {len(users):,} 중 {len(U):,}명 채점 · 이력 길이 중앙 {int(U['hist'].str.len().median())}"
      f" · 상품끼리 코사인 평균 {float((E[:500] @ E[:500].T).mean()):.3f} → 공통 성분 제거 후 {float((Ec[:500] @ Ec[:500].T).mean()):.3f}")


def rank_of(scores, hist, target):
    s = scores.copy(); s[hist] = -np.inf
    return int((s > s[target]).sum()) + 1


def ranks_for(profile, UU, X=None):
    """사용자 200명씩 : 프로필 벡터들 × 전체 상품(X, 기본 E) → 사용자마다 최댓값 → 가려 둔 상품의 순위"""
    X = E if X is None else X
    out = []
    for s in range(0, len(UU), 200):
        ch = UU.iloc[s:s + 200]
        Ps = [np.atleast_2d(profile(h)) for h in ch["hist"]]
        S = np.maximum.reduceat(unit(np.vstack(Ps)) @ X.T, np.r_[0, np.cumsum([len(p) for p in Ps])[:-1]], axis=0)
        for row, h, tg in zip(S, ch["hist"], ch["target"]):
            row[h] = -np.inf; out.append(int((row > row[tg]).sum()) + 1)
    return np.array(out)


RK = {}                                                            # 방식 이름 → 사용자별 순위 (셀 10)


def score_users(profile, name, X=None):
    """profile(hist) → (m, 4096) 프로필 벡터들. 후보는 X(기본 E). 사용자별 순위를 RK 에 두고 Hit@10 · MRR 반환"""
    r = RK[name] = ranks_for(profile, U, X)
    return {"방식": name, "Hit@10": round(float((r <= 10).mean()), 4), "MRR": round(float(np.where(r <= 10, 1 / r, 0).mean()), 4)}

### 셀 8 · 추천 SQL — 한 사용자
취향 벡터 = 이력 상품 벡터의 평균. pgvector 의 `avg(embedding)` 가 원소별 평균이다.
그 벡터와 코사인 거리(`<=>`)가 작은 순 = 추천. 이미 산 상품은 빼고, 가려 둔 실제 다음 구매가 몇 위인지 본다.
- 취향 벡터 부분질의는 바깥 행과 상관이 없어 한 번만 계산된다
- 여기서는 4096차원 원본으로 상품을 전부 비교한다(인덱스 없음)
- 끝에 **공통 성분 제거** 판 : 상품 벡터에서 전체 평균 μ 를 빼고(`embedding - μ`, `l2_normalize`) 평균 → 같은 사용자의 실제 다음 구매 순위

In [ ]:
# 셀 8 · 추천 SQL
u = U[U["hist"].str.len().between(4, 8)].iloc[0]
print("사용자 :", u["user_id"])
display(sql("""
    SELECT p.seq, i.title AS 구매_이력, i.category AS 분류
    FROM lab3.shop_purchases p JOIN lab3.shop_items i ON i.id = p.item_id
    WHERE p.user_id = %s AND NOT p.is_target ORDER BY p.seq""", (u["user_id"],)))

PROFILE = """(SELECT avg(i.embedding) FROM lab3.shop_purchases p JOIN lab3.shop_items i ON i.id = p.item_id
              WHERE p.user_id = %(u)s AND NOT p.is_target)"""
RECOMMEND = f"""
    SELECT i.title AS 추천, i.category AS 분류, round((1 - (i.embedding <=> {PROFILE}))::numeric, 3) AS 코사인,
           i.id = (SELECT item_id FROM lab3.shop_purchases WHERE user_id = %(u)s AND is_target) AS 실제_다음_구매
    FROM lab3.shop_items i
    WHERE i.id NOT IN (SELECT item_id FROM lab3.shop_purchases WHERE user_id = %(u)s AND NOT is_target)
    ORDER BY i.embedding <=> {PROFILE}
    LIMIT 10"""
print(f"‖M‖ = {one(f'SELECT vector_norm({PROFILE})', {'u': u['user_id']}):.3f}")
t = time.time()
display(sql(RECOMMEND, {"u": u["user_id"]}))
print(f"SQL {time.time() - t:.2f}s")
M = E[u["hist"]].mean(axis=0)
print("실제 다음 구매 :", items.loc[u["target"], "title"], "→", rank_of(E @ unit(M), u["hist"], u["target"]), "위 (행렬 연산으로 확인)")

CENTER = "(SELECT avg(embedding) FROM lab3.shop_items)"          # μ : 상품 전체 평균 (공통 성분)
CPROFILE = f"""(SELECT avg(l2_normalize(i.embedding - {CENTER})) FROM lab3.shop_purchases p JOIN lab3.shop_items i
               ON i.id = p.item_id WHERE p.user_id = %(u)s AND NOT p.is_target)"""
crank = one(f"""
    WITH ranked AS (
        SELECT i.id, row_number() OVER (ORDER BY (i.embedding - {CENTER}) <=> {CPROFILE}) AS rk
        FROM lab3.shop_items i
        WHERE i.id NOT IN (SELECT item_id FROM lab3.shop_purchases WHERE user_id = %(u)s AND NOT is_target))
    SELECT rk FROM ranked WHERE id = (SELECT item_id FROM lab3.shop_purchases WHERE user_id = %(u)s AND is_target)""", {"u": u["user_id"]})
print(f"공통 성분 제거 : 실제 다음 구매 → SQL {crank} 위 · 행렬 연산 {rank_of(Ec @ unit(Ec[u['hist']].mean(axis=0)), u['hist'], u['target'])} 위")

### 셀 9 · 추천 API — HNSW 후보 → 원본 재정렬
서비스에서는 매번 상품 전부를 4096차원으로 비교하지 않는다. 4-1 실습5 셀 6 과 같이
`emb128` HNSW 인덱스로 후보 N개를 뽑고 → 그 안에서만 원본 4096 으로 재정렬한다.
- `hnsw.ef_search` 는 후보 수 이상으로(HNSW 는 ef_search 개까지만 돌려준다)
- 사용자 `N_API`명에게 전수 SQL(셀 8 방식) · 후보 N 별 SQL 로 추천해 Hit@10 · 한 건 시간을 잰다
- 전수 SQL 의 Hit@10 은 같은 사용자를 행렬 연산으로 채점한 값(마지막 줄)과 같아야 한다(같은 계산)

In [ ]:
# 셀 9 · 추천 API
API = """
    WITH prof AS MATERIALIZED (
        SELECT avg(i.embedding) AS m FROM lab3.shop_purchases p JOIN lab3.shop_items i ON i.id = p.item_id
        WHERE p.user_id = %(u)s AND NOT p.is_target)
    SELECT c.id FROM (
        SELECT i.id, i.embedding FROM lab3.shop_items i
        ORDER BY i.emb128 <=> (SELECT l2_normalize(subvector(m, 1, 128))::vector(128) FROM prof)
        LIMIT %(n)s) c
    WHERE c.id NOT IN (SELECT item_id FROM lab3.shop_purchases WHERE user_id = %(u)s AND NOT is_target)
    ORDER BY c.embedding <=> (SELECT m FROM prof)
    LIMIT 10"""
plan = "\n".join(r[0] for r in conn.execute("EXPLAIN " + API.replace("%(u)s", "'x'").replace("%(n)s", "100")).fetchall())
print("후보 추리기에 쓰는 인덱스 :", [w for w in plan.split() if "hnsw" in w] or "없음(전수)")

UA = U.iloc[:N_API]
gold = dict(zip(UA["user_id"], items["id"].values[UA["target"].values]))
rows, exact = [], []
t = time.time()
for uid in UA["user_id"]:
    ids = [r[0] for r in conn.execute(f"""
        SELECT i.id FROM lab3.shop_items i
        WHERE i.id NOT IN (SELECT item_id FROM lab3.shop_purchases WHERE user_id = %(u)s AND NOT is_target)
        ORDER BY i.embedding <=> {PROFILE} LIMIT 10""", {"u": uid}).fetchall()]
    exact.append(gold[uid] in ids)
rows.append({"방식": "전수 4096 (인덱스 없음)", "후보 N": "-", "Hit@10": np.mean(exact), "한 건(ms)": (time.time() - t) / len(UA) * 1000})
for n in CANDIDATES:
    conn.execute(f"SET hnsw.ef_search = {max(40, n)}")
    t = time.time(); h = []
    for uid in UA["user_id"]:
        h.append(gold[uid] in [r[0] for r in conn.execute(API, {"u": uid, "n": n}).fetchall()])
    rows.append({"방식": "emb128 HNSW 후보 → 4096 재정렬", "후보 N": n, "Hit@10": np.mean(h), "한 건(ms)": (time.time() - t) / len(UA) * 1000})
conn.execute("RESET hnsw.ef_search")
r = ranks_for(lambda h: E[h].mean(axis=0), UA)
rows.append({"방식": "행렬 연산(같은 사용자, 채점용)", "후보 N": "-", "Hit@10": float((r <= 10).mean()), "한 건(ms)": None})
R9 = pd.DataFrame(rows).round(4)
R9

### 셀 10 · 추천 방식 비교 (행렬 연산)
마지막 열 = 평균(취향 벡터) 대비 Hit@10 차이와 사용자 단위 95% 신뢰구간(✔ = 우연으로 보기 어려움).
| 방식 | 프로필 |
|---|---|
| 인기순 | 개인화 없음 — 표본 사용자 이력에 많이 나온 상품 |
| 마지막 구매 | 가장 최근 상품 하나 |
| 평균(취향 벡터) | 이력 전체의 평균 |
| 최근 가중 평균 | 최근 구매일수록 큰 가중치 |
| 분류별 평균 | 이력을 상품 분류(플랫폼 등)로 나눠 분류마다 평균 → 가장 가까운 것 (장표 20 "카테고리별 평균") |
| … · 공통 성분 제거 | 같은 계산을 `Ec`(전체 평균을 뺀 상품 벡터)로 : 프로필도 후보도 Ec |

- 인기순은 이 테이블의 구매 이력으로 센다 → 표본 사용자가 적으면(기본 500명) 인기 신호도 약하다

In [ ]:
# 셀 10 · 추천 방식 비교
t = time.time()
R10 = []
r = RK["인기순"] = np.array([rank_of(pop, h, tg) for h, tg in zip(U["hist"], U["target"])])
R10.append({"방식": "인기순", "Hit@10": round(float((r <= 10).mean()), 4), "MRR": round(float(np.where(r <= 10, 1 / r, 0).mean()), 4)})
R10.append(score_users(lambda h: E[h[-1]], "마지막 구매"))
R10.append(score_users(lambda h: E[h].mean(axis=0), "평균(취향 벡터)"))


def recent(h):
    w = 0.5 ** (np.arange(len(h))[::-1] / RECENT_HALF)
    return (E[h] * w[:, None]).sum(axis=0)


def recent_c(h):
    w = 0.5 ** (np.arange(len(h))[::-1] / RECENT_HALF)
    return (Ec[h] * w[:, None]).sum(axis=0)


R10.append(score_users(recent, "최근 가중 평균"))
R10.append(score_users(lambda h: np.stack([E[[i for i in h if lab[i] == c]].mean(axis=0) for c in set(lab[h])]), "분류별 평균"))
R10.append(score_users(lambda h: Ec[h].mean(axis=0), "평균 · 공통 성분 제거", Ec))
R10.append(score_users(recent_c, "최근 가중 · 공통 성분 제거", Ec))
R10.append(score_users(lambda h: np.stack([Ec[[i for i in h if lab[i] == c]].mean(axis=0) for c in set(lab[h])]), "분류별 평균 · 공통 성분 제거", Ec))
R10 = pd.DataFrame(R10).set_index("방식")
base = RK["평균(취향 벡터)"] <= 10
R10["Hit@10 − 평균 [95% 구간]"] = [ci((RK[m] <= 10).astype(float) - base) if m != "평균(취향 벡터)" else "" for m in R10.index]
hit = lambda m: (RK[m] <= 10).astype(float)
print(f"사용자 {len(U):,}명 · 후보 상품 {len(items):,}개 · {time.time() - t:.0f}s")
print("평균 · 공통 성분 제거 − 마지막 구매", ci(hit("평균 · 공통 성분 제거") - hit("마지막 구매")),
      "· − 인기순", ci(hit("평균 · 공통 성분 제거") - hit("인기순")))
R10

### 셀 11 · 시간 순서를 뺀 채점 — 이력 중 무작위 하나를 가림
셀 10 은 **마지막** 구매를 가린다 → 직전 구매가 유리하다(기기를 사고 그 액세서리를 잇달아 삼). 평균이 취향을 대표하는지만 보려면 시간 순서를 뺀다.
- 사용자마다 (이력 + 마지막 구매) 중 무작위 하나를 가리고 나머지로 추천 · 인기순은 가린 상품을 이 사용자 몫만큼 빼고 센다
- 비교 : 남은 이력 중 **무작위 한 개**로 추천 vs 남은 이력 **전부의 평균** — 평균이 한 개보다 나으면 "여러 구매의 평균 = 취향"

In [ ]:
# 셀 11 · 무작위 하나를 가림
rng11 = np.random.default_rng(SEED)
held = []
for h, tg in zip(U["hist"], U["target"]):
    full = h + [tg]; j = int(rng11.integers(len(full)))
    held.append((full[:j] + full[j + 1:], full[j], j < len(h)))       # (남은 이력, 가린 상품, 가린 것이 이력 안의 상품인가)
U11 = pd.DataFrame([(a, b) for a, b, _ in held], columns=["hist", "target"])
pick = [int(rng11.integers(len(h))) for h in U11["hist"]]             # "무작위 한 개" 에 쓸 이력 위치 (미리 뽑아 고정)
one_of = {tuple(h): h[k] for h, k in zip(U11["hist"], pick)}
R11r = {"인기순": np.array([rank_of(pop - np.eye(1, len(pop), tg)[0] * inh, h, tg) for h, tg, inh in held]),
        "이력 중 무작위 한 개": ranks_for(lambda h: E[one_of[tuple(h)]], U11),
        "평균(취향 벡터)": ranks_for(lambda h: E[h].mean(axis=0), U11),
        "무작위 한 개 · 공통 성분 제거": ranks_for(lambda h: Ec[one_of[tuple(h)]], U11, Ec),
        "평균 · 공통 성분 제거": ranks_for(lambda h: Ec[h].mean(axis=0), U11, Ec)}
h11 = {k: (v <= 10).astype(float) for k, v in R11r.items()}
R11b = pd.DataFrame({"Hit@10": {k: round(v.mean(), 4) for k, v in h11.items()}})
print("평균 − 무작위 한 개 :", ci(h11["평균(취향 벡터)"] - h11["이력 중 무작위 한 개"]),
      "· 공통 성분 제거 후 :", ci(h11["평균 · 공통 성분 제거"] - h11["무작위 한 개 · 공통 성분 제거"]),
      "· 평균(제거) − 인기순 :", ci(h11["평균 · 공통 성분 제거"] - h11["인기순"]))
R11b

### 셀 12 · ‖M‖(응집도)과 추천 적중
장표 18 : 길이 1 벡터들의 평균은 방향이 모일수록 길다 → ‖M‖ 이 작으면 관심사가 흩어진 사용자.
- 사용자를 ‖M‖ 로 5등분해 방식별 Hit@10 을 본다(표본이 작으면 구간마다 사용자가 적어 흔들린다)
- 첫 줄 : `‖M‖² = 이력 안 모든 쌍 코사인의 평균(자기 자신 포함)` 을 한 사용자로 확인

In [ ]:
# 셀 12 · ‖M‖ 과 추천 적중
h = U["hist"].iloc[0]
print(f"‖M‖² = {np.linalg.norm(E[h].mean(axis=0)) ** 2:.6f} · 쌍 코사인 평균 = {(E[h] @ E[h].T).mean():.6f}")
U["norm"] = [float(np.linalg.norm(E[h].mean(axis=0))) for h in U["hist"]]
U["q"] = pd.qcut(U["norm"], 5, labels=["1 흩어짐", "2", "3", "4", "5 모임"])
rows = []
for q, g in U.groupby("q", observed=True):
    r = {}
    for name, f in [("마지막 구매", lambda h: E[h[-1]]), ("평균", lambda h: E[h].mean(axis=0)),
                    ("분류별 평균", lambda h: np.stack([E[[i for i in h if lab[i] == c]].mean(axis=0) for c in set(lab[h])]))]:
        r[name] = round(float((ranks_for(f, g) <= 10).mean()), 4)
    rows.append({"‖M‖ 구간": q, "사용자": len(g), "‖M‖ 범위": f"{g['norm'].min():.2f}~{g['norm'].max():.2f}", **r})
R12 = pd.DataFrame(rows).set_index("‖M‖ 구간")
R12[["마지막 구매", "평균", "분류별 평균"]].plot(marker="o", figsize=(9, 3.6), title="응집도(‖M‖) 구간별 Hit@10")
plt.ylabel("Hit@10"); plt.show()
R12

### 셀 13 · 중심 벡터 : 코사인이든 내적이든 순위는 같다
장표 18 : 중심 M 과의 코사인 = 내적 ÷ ‖M‖. 한 사용자 안에서 ‖M‖ 은 모든 후보에 같으므로 순위가 그대로다.
SQL 로 바꾸면 `ORDER BY embedding <=> M`(코사인 거리) 와 `ORDER BY embedding <#> M`(음의 내적) 이 같은 순서라는 뜻 → 셀 끝에서 셀 8 사용자로 확인.

In [ ]:
# 셀 13 · 코사인 vs 내적 순위
same, gap = 0, 0.0
UU = U["hist"].iloc[:500]
for h in UU:
    M = E[h].mean(axis=0)
    sd, sc = E @ M, E @ unit(M)                                    # 내적 · 코사인(후보는 길이 1)
    a, b = np.argsort(-sd, kind="stable")[:100], np.argsort(-sc, kind="stable")[:100]
    if (a == b).all():
        same += 1
    else:                                                          # 순서가 다른 첫 자리 : 두 상품의 코사인 차
        k = np.flatnonzero(a != b)[0]; gap = max(gap, abs(sc[a[k]] - sc[b[k]]))
print(f"사용자 {len(UU)}명 중 상위 100 순서가 같은 사람 {same}명",
      f"· 다른 {len(UU) - same}명은 바뀐 자리의 코사인 차가 최대 {gap:.1e} (float32 반올림 수준의 동점)" if same < len(UU) else "")
a, b = [[r[0] for r in conn.execute(f"SELECT id FROM lab3.shop_items ORDER BY embedding {op} {PROFILE} LIMIT 20",
                                    {"u": u["user_id"]}).fetchall()] for op in ["<=>", "<#>"]]
print("SQL (셀 8 사용자) : <=> 상위 20 과 <#> 상위 20 이 같은가 →", a == b)

### 셀 14 · 정리
- 셀 9 : HNSW 후보 몇 개면 전수와 같은 적중인가 · 한 건 시간
- 셀 10 : 원래 벡터의 평균은 약하다 — 상품 벡터가 한쪽으로 쏠려 있어 평균이 공통 방향을 가리킨다. **공통 성분을 빼고 평균**하면 크게 오른다 · 다음 구매 하나를 맞히는 데는 직전 구매도 강하다(연속 구매)
- 셀 11 : 시간 순서를 빼면(무작위 하나를 가림) 공통 성분을 뺀 평균이 한 개보다 낫다 = 여러 구매의 평균이 취향을 대표
- 셀 12 : 흩어진 사용자에게 분류별 평균이 도움이 되는가
- 표본 사용자가 적으면 숫자가 흔들린다 → 셀 1 `N_USERS` 를 키우고 `RESET = True` 로 다시(이미 계산한 벡터는 캐시에서)

In [ ]:
# 셀 14 · 정리
print(f"표본 사용자 {len(users):,}명 · 상품 {len(items):,}개")
display(R9)
display(R10)
display(R11b)
R12